In [24]:
import os
os.environ["HF_HOME"] = "/kaggle/temp/hf"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("fitcheck")
    print("HF token loaded")
except Exception as e:
    print("No HF token:", e)

HF token loaded


In [25]:
!rm -rf /kaggle/working/fitcheck
!git clone -q https://github.com/Anassbzdd/fitcheck.git /kaggle/working/fitcheck
!cd /kaggle/working/fitcheck && pip install -q -e . && pip install -q -r scripts/requirements-measure.txt
!ls /kaggle/working/fitcheck/scripts/

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fitcheck-llm (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.2 MB/s eta 0:00:00:00:01
measure_infer.py  measure.py  requirements-measure.txt


In [26]:
!python -c "import torch, peft, transformers, torchao; print('torch:', torch.__version__); print('peft:', peft.__version__); print('transformers:', transformers.__version__); print('torchao:', torchao.__version__)"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
torch: 2.10.0+cu128
peft: 0.19.1
transformers: 5.0.0
torchao: 0.18.0


In [27]:
!cd /kaggle/working/fitcheck && python scripts/measure.py JackFram/llama-160m --quant none --precision fp16 --lora-r 32 --batch-size 4 --seq-len 1024 --gpu t4 2>&1 | tee /kaggle/working/A1_eager.txt

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 111/111 [00:00<00:00, 587.74it/s, Materializing param=model.norm.weight]                              

  JackFram/llama-160m  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=4, seq=1024, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 164,776,704 logical | 2,359,296 trainable
          base 162,417,408 vs fitcheck P 162,417,792  (-384)  [OK]
  optimizer states: 18 MiB observed, dtype float32
  step time: 0.44s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      328 MiB
    gradients (after backward)                  9 MiB
    peak allocated  (tensor bytes)          7,537 MiB
    peak reserved   (allocator 

In [28]:
!cd /kaggle/working/fitcheck && python scripts/measure.py JackFram/llama-160m --quant none --precision fp16 --lora-r 32 --batch-size 4 --seq-len 1024 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/A1_sdpa.txt

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 111/111 [00:00<00:00, 750.42it/s, Materializing param=model.norm.weight]                              

  JackFram/llama-160m  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=4, seq=1024, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 164,776,704 logical | 2,359,296 trainable
          base 162,417,408 vs fitcheck P 162,417,792  (-384)  [OK]
  optimizer states: 18 MiB observed, dtype float32
  step time: 0.30s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocate

In [29]:
!cd /kaggle/working/fitcheck && python scripts/measure.py unsloth/Llama-3.2-1B --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512 --gpu t4 2>&1 | tee /kaggle/working/A2_eager.txt

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 146/146 [00:01<00:00, 126.46it/s, Materializing param=model.norm.weight]                              

  unsloth/Llama-3.2-1B  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,242,630,144 logical | 6,815,744 trainable
          base 1,235,814,400 vs fitcheck P 1,235,814,400  (+0)  [OK]
  optimizer states: 52 MiB observed, dtype float32
  step time: 0.43s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    2,384 MiB
    gradients (after backward)                 26 MiB
    peak allocated  (tensor bytes)          7,481 MiB
    peak reserved   (alloca

In [30]:
!cd /kaggle/working/fitcheck && python scripts/measure.py unsloth/Llama-3.2-1B --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/A2_sdpa.txt

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 146/146 [00:01<00:00, 125.60it/s, Materializing param=model.norm.weight]                              

  unsloth/Llama-3.2-1B  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,242,630,144 logical | 6,815,744 trainable
          base 1,235,814,400 vs fitcheck P 1,235,814,400  (+0)  [OK]
  optimizer states: 52 MiB observed, dtype float32
  step time: 0.37s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allo

In [31]:
!cd /kaggle/working/fitcheck && python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512 --gpu t4 2>&1 | tee /kaggle/working/A3_eager.txt

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 203.30it/s, Materializing param=model.norm.weight]                              down_proj.weight]  

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,109,059,584 logical | 9,011,200 trainable
          base 1,100,048,384 vs fitcheck P 1,100,048,384  (+0)  [OK]
  optimizer states: 69 MiB observed, dtype float32
  step time: 0.46s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    2,135 MiB
    gradients (after backward)                 34 MiB
    peak allocated  (tensor bytes)          6,87

In [32]:
!cd /kaggle/working/fitcheck && python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/A3_sdpa.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 188.33it/s, Materializing param=model.norm.weight]                              izing param=model.layers.17.mlp.down_proj.weight]

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,109,059,584 logical | 9,011,200 trainable
          base 1,100,048,384 vs fitcheck P 1,100,048,384  (+0)  [OK]
  optimizer states: 69 MiB observed, dtype float32
  step time: 0.38s

  MEASURED
    CUDA context (at peak)                    141 MiB

In [33]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-135M --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --gpu t4 2>&1 | tee /kaggle/working/A4_eager.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1117.33it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-135M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=1024, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 138,201,408 logical | 3,686,400 trainable
          base 134,515,008 vs fitcheck P 134,515,008  (+0)  [OK]
  optimizer states: 28 MiB observed, dtype float32
  step time: 0.37s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      271 MiB
    gradients (after backward)                 14 MiB
    peak allocated  (tensor bytes)          6,490 MiB
    peak reserved   (allo

In [34]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-135M --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/A4_sdpa.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1147.94it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-135M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=1024, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 138,201,408 logical | 3,686,400 trainable
          base 134,515,008 vs fitcheck P 134,515,008  (+0)  [OK]
  optimizer states: 28 MiB observed, dtype float32
  step time: 0.23s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    al

In [35]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-360M --quant none --precision fp16 --lora-r 32 --batch-size 4 --seq-len 512 --gpu t4 2>&1 | tee /kaggle/working/A5_eager.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 702.79it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-360M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=4, seq=512, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 368,374,720 logical | 6,553,600 trainable
          base 361,821,120 vs fitcheck P 361,821,120  (+0)  [OK]
  optimizer states: 50 MiB observed, dtype float32
  step time: 0.54s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      729 MiB
    gradients (after backward)                 25 MiB
    peak allocated  (tensor bytes)          7,812 MiB
    peak reserved   (alloca

In [36]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-360M --quant none --precision fp16 --lora-r 32 --batch-size 4 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/A5_sdpa.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 809.01it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-360M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=4, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 368,374,720 logical | 6,553,600 trainable
          base 361,821,120 vs fitcheck P 361,821,120  (+0)  [OK]
  optimizer states: 50 MiB observed, dtype float32
  step time: 0.42s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allo

In [37]:
!cd /kaggle/working/fitcheck && python scripts/measure_infer.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --precision fp16 --seq-len 2048 --num-concurrent 1 --gpu t4 2>&1 | tee /kaggle/working/B1_serve.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 178.35it/s, Materializing param=model.norm.weight]                              

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  serving on  Tesla T4
  serve seq=2048, concurrent=1, fp16, quant=none

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  params: 1,100,048,384 logical
  cached tokens/sequence: 2,048 (asked for 2,048)   kv dtype: float16
  prefill: 0.89s   decode: 51.9 ms/token

  MEASURED
    CUDA context (at peak)                    133 MiB
    allocated after load                    2,100 MiB
    KV cache, walked directly                  44 MiB
    KV cache, by subtraction                   52 MiB
    resident after fill                     2,152 MiB
    peak allocated  (tensor bytes)          2,205 MiB
    peak reserved   (allocator pool)        2,322 MiB
    process total   (reserved+ctx)   

In [38]:
!cd /kaggle/working/fitcheck && python scripts/measure_infer.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --precision fp16 --seq-len 2048 --num-concurrent 16 --gpu t4 2>&1 | tee /kaggle/working/B2_serve.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 199.02it/s, Materializing param=model.norm.weight]                              

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  serving on  Tesla T4
  serve seq=2048, concurrent=16, fp16, quant=none

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  params: 1,100,048,384 logical
  cached tokens/sequence: 2,048 (asked for 2,048)   kv dtype: float16
  prefill: 6.54s   decode: 216.5 ms/token

  MEASURED
    CUDA context (at peak)                    133 MiB
    allocated after load                    2,100 MiB
    KV cache, walked directly                 704 MiB
    KV cache, by subtraction                  712 MiB
    resident after fill                     2,812 MiB
    peak allocated  (tensor bytes)          3,650 MiB
    peak reserved   (allocator pool)        4,388 MiB
    process total   (reserved+ctx) 

In [39]:
!cd /kaggle/working/fitcheck && python scripts/measure_infer.py Qwen/Qwen2.5-1.5B --precision fp16 --seq-len 2048 --num-concurrent 8 --gpu t4 2>&1 | tee /kaggle/working/B3_serve.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 233.24it/s, Materializing param=model.norm.weight]                              

  Qwen/Qwen2.5-1.5B  serving on  Tesla T4
  serve seq=2048, concurrent=8, fp16, quant=none

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  params: 1,543,714,304 logical
  cached tokens/sequence: 2,048 (asked for 2,048)   kv dtype: float16
  prefill: 4.27s   decode: 130.2 ms/token

  MEASURED
    CUDA context (at peak)                    133 MiB
    allocated after load                    2,945 MiB
    KV cache, walked directly                 448 MiB
    KV cache, by subtraction                  457 MiB
    resident after fill                     3,402 MiB
    peak allocated  (tensor bytes)          3,723 MiB
    peak reserved   (allocator pool)        4,270 MiB
    process total   (reserved+ctx)          4,403 MiB

In [40]:
!cd /kaggle/working/fitcheck && python scripts/measure_infer.py HuggingFaceTB/SmolLM2-1.7B --quant nf4 --precision fp16 --seq-len 2048 --num-concurrent 4 --gpu t4 2>&1 | tee /kaggle/working/B4_serve.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 218/218 [00:02<00:00, 93.36it/s, Materializing param=model.norm.weight]                               

  HuggingFaceTB/SmolLM2-1.7B  serving on  Tesla T4
  serve seq=2048, concurrent=4, fp16, quant=nf4

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  params: 1,711,376,384 logical
  cached tokens/sequence: 2,048 (asked for 2,048)   kv dtype: float16
  prefill: 3.14s   decode: 71.4 ms/token

  MEASURED
    CUDA context (at peak)                    133 MiB
    allocated after load                    1,056 MiB
    KV cache, walked directly               1,536 MiB
    KV cache, by subtraction                1,544 MiB
    resident after fill                     2,600 MiB
    peak allocated  (tensor bytes)          2,682 MiB
    peak reserved   (allocator pool)        2,742 MiB
    process total   (reserved+ctx)          2,

In [41]:
!cd /kaggle/working/fitcheck && python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --quant int8 --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --grad-checkpoint --gpu t4 2>&1 | tee /kaggle/working/C1_int8.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 201/201 [00:02<00:00, 68.22it/s, Materializing param=model.norm.weight]                               
MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.float32 to float16 du

In [42]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-1.7B --quant nf4 --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --grad-checkpoint --gpu t4 2>&1 | tee /kaggle/working/C2_nodq.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 218/218 [00:02<00:00, 106.93it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-1.7B  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=1024, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,723,959,296 logical | 12,582,912 trainable
          base 1,711,376,384 vs fitcheck P 1,711,376,384  (+0)  [OK]
  optimizer states: 96 MiB observed, dtype float32
  step time: 4.12s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    1,296 MiB
    gradients (after backward)                 48 MiB
    peak allocated  (tensor bytes)          3,353 MiB
    peak res

In [43]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-1.7B --quant nf4 --double-quant --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --grad-checkpoint --gpu t4 2>&1 | tee /kaggle/working/C3_dq.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 218/218 [00:02<00:00, 104.02it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-1.7B  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=1024, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,723,959,296 logical | 12,582,912 trainable
          base 1,711,376,384 vs fitcheck P 1,711,376,384  (+0)  [OK]
  optimizer states: 96 MiB observed, dtype float32
  step time: 3.83s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    1,225 MiB
    gradients (after backward)                 48 MiB
    peak allocated  (tensor bytes)          3,282 MiB
    peak res

In [44]:
!cd /kaggle/working/fitcheck && python scripts/measure.py JackFram/llama-160m --no-lora --quant none --precision fp16 --batch-size 2 --seq-len 1024 --grad-checkpoint --gpu t4 2>&1 | tee /kaggle/working/D1_fullft.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 111/111 [00:00<00:00, 771.98it/s, Materializing param=model.norm.weight]                              

  JackFram/llama-160m  on  Tesla T4
  full FT, bs=2, seq=1024, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 162,417,408 logical | 162,417,408 trainable
          base 0 vs fitcheck P 162,417,792  (-162,417,792)  [MISMATCH]
          ^ config_parser's param count disagrees with the loaded model; every per-param term is built on P, so fix this first.
  optimizer states: 1,239 MiB observed, dtype float32
  step time: 1.15s

  MEASURED
    CUDA context (at peak)                    139 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      319 MiB
    grad

In [45]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-135M --quant none --precision fp32 --lora-r 32 --batch-size 2 --seq-len 1024 --grad-checkpoint --gpu t4 2>&1 | tee /kaggle/working/D2_fp32.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 978.36it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-135M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=1024, fp32, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 138,201,408 logical | 3,686,400 trainable
          base 134,515,008 vs fitcheck P 134,515,008  (+0)  [OK]
  optimizer states: 28 MiB observed, dtype float32
  step time: 1.22s

  MEASURED
    CUDA context (at peak)                    139 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      535 MiB
    gradients (after backward)                 14 MiB
    peak allocated  (tensor bytes)          2,264 MiB
    peak reserved   

In [46]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-135M --quant none --precision fp32 --lora-r 32 --batch-size 2 --seq-len 1024 --grad-checkpoint --gpu t4 2>&1 | tee /kaggle/working/D2_fp32.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1071.87it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-135M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=1024, fp32, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 138,201,408 logical | 3,686,400 trainable
          base 134,515,008 vs fitcheck P 134,515,008  (+0)  [OK]
  optimizer states: 28 MiB observed, dtype float32
  step time: 1.26s

  MEASURED
    CUDA context (at peak)                    139 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      535 MiB
    gradients (after backward)                 14 MiB
    peak allocated  (tensor bytes)          2,264 MiB
    peak reserved  